In [29]:
import pandas as pd
from functools import reduce
import numpy as np

In [19]:
config = {
    'shift_start_time': '18:00:00',
    'shift_end_time': '02:00:00'
}

In [ ]:
start_time = pd.to_datetime(config['shift_start_time']).time()
end_time = pd.to_datetime(config['shift_end_time']).time()
shift_T = (end_time.hour - start_time.hour) % 24

8

In [5]:
readings_df = pd.read_csv('noise_data.csv')
readings_df['timestamp'] = pd.to_datetime(readings_df['timestamp'])

In [26]:
if start_time <= end_time:
    mask = (readings_df['timestamp'].dt.time >= start_time) & (readings_df['timestamp'].dt.time <= end_time)
else: # overnight shift
    mask = (readings_df['timestamp'].dt.time >= start_time) | (readings_df['timestamp'].dt.time <= end_time)

shift_readings_df = readings_df[mask]
shift_readings_df

,timestamp,value
0,2009-09-09 00:02:27,81
9,2009-09-09 22:45:18,31
10,2009-10-09 00:31:24,76
11,2009-10-09 00:52:11,14
12,2009-10-09 00:53:17,45
28,2009-11-09 18:16:04,30
29,2009-11-09 18:56:36,45
30,2009-12-09 00:00:24,16
31,2009-12-09 01:23:51,89
32,2009-12-09 01:38:01,45


In [43]:
shift_readings_df = shift_readings_df.copy()
shift_readings_df['energy'] = 10 ** (shift_readings_df['value'] / 10)

daily_exp_lvl_df = shift_readings_df \
    .groupby(shift_readings_df['timestamp'].dt.date) \
    .aggregate({"energy": "mean"}) 

daily_exp_lvl_df['leq'] = 10*np.log10(daily_exp_lvl_df["energy"]*(shift_T/8)) # adjust to 8-hour TWA and turn back to dB

daily_exp_lvl_df

,energy,leq
timestamp,,
2009-08-09,2.505986e+09,93.989787
2009-09-09,6.294690e+07,77.989743
2009-10-09,1.328079e+07,71.232239
2009-11-09,1.631139e+04,42.124909
2009-12-09,2.647866e+08,84.228961
